# Fields

In [ ]:
import mefikit as mf
import numpy as np
import pyvista as pv

pv.set_plot_theme("dark")
pv.set_jupyter_backend("static")

## Field expressions


FieldExpr are composition of floats and mf.sel.field("fieldname") or custom fields :
- field("toto") * field("tata")
- field("toto") + field("tata")
- field("toto") - field("tata")
- field("toto") / field("tata")
- field("toto") ** field("tata")
- field("toto").dot(field("tata"))
- field("toto") @ field("tata")
- sin(field("toto"))
- cos(field("toto"))
- abs(field("toto"))
- log(field("toto"))
- log10(field("toto"))
- exp(field("toto"))
- field("toto")[0]
- normals()
- x()
- y()
- z()
- centroids()

In [ ]:
x = np.logspace(-5, 0.0, 1000)
mesh2 = mf.build_cmesh(x, x)

In [ ]:
mesh2.measure_update()

In [ ]:
mesh2.to_pyvista().plot()

In [ ]:
m = mf.Field("Measure")

## Field operations evaluation

In [ ]:
mesh2.eval_update("4 * M2", m * m * 4.0)

In [ ]:
mesh2.to_pyvista().plot()

In [ ]:
mesh2.fields()

In [ ]:
m2 = mf.Field("4 * M2")
mesh2.eval(m2 - 4.0 * m.square())

## How does it work ?

In [ ]:
print(4.0 * m * m)

## Why is it awesome ?

This enables two patterns :
- reusability and composition of filters
- evaluation optimizations, some selection filters are evaluated in parallel, some are evaluated first if they are discriminant

# Field to Selection

Fields can be converted to threshold selections :

In [ ]:
th = (m > 3.25e-5) & (m < 1e-4)

In [ ]:
m2sel = mesh2.select(th)
pvm2: pv.UnstructuredGrid = m2sel.to_pyvista()
pvm2.active_scalars_name = "Measure"
pvm2.plot()

Thoses threasholds selections can be combined with other selections.

In [ ]:
r = mf.sel.rect([0.25, 0.25], [0.7, 0.7])
c = mf.sel.circle([0.875, 0.875], 0.05)

In [ ]:
mesh2.select((m2 > 4e-9) - r - c).to_pyvista().plot()

## Transfering fields

The transfer function can be
- interpolation
- extrapolation
- conservative
- non-conservative
- using cells
- using cell centers and point clouds methods
- etc

They are many.

### ConstantPiecewise Transfer

The transfer is very simple. It is based on the cells of src_mesh and the cells center of the target mesh. It assigns to a cell from the target the value of the cell in which the center is located in. This is a point location based value assignment. By default the centroid (mean of cell nodes) is used because it is fast to comupute and is accurate with regular cells.

In [ ]:
m_src = mesh2.select((m2 > 4e-9) - r - c)
m_tgt = mf.build_cmesh(np.linspace(0.5, 1.5, 20), np.linspace(0.0, 1.5, 20))

# The transfer is computed between source and target geometry.
# This step is computationnaly heavy, but done once.
cpt = mf.transfer.ConstantPiecewise(m_src, m_tgt)

In [ ]:
# The transfer is applied. This step is much faster.
cpt.apply_update(m_src, "Measure", m_tgt, tgt_field_name="Projection", def_val=np.nan)

In [ ]:
m_tgt.to_pyvista().plot(show_edges=True)

As you can see the `"Measure"` field from m_src was used to compute the `"Projection"` field on m_tgt. Both mesh are not completly overlapping but that is not an issue. Cells from m_tgt whose center is not in a cell from m_src take a default value `def_val`. Default is 0.0 but any floating point value, such as `np.nan` is accepted.

This interpolation is good when coarseing a mesh and you do not need conservation. It might be useful in other circumstances I do not know of. It is quite fast but not that much because of the `is_in_cell` exact geometrical query.

### MovingMean Transfer

This Transfer is based on m_src cell center positions and m_tgt cell centers positions. It is a "meshless" operation as it does not care about connectivity. There are several options :

- normal mean
- weighted mean

Pros :

- it is extremly fast
- it does not overshoot / undershoot

Cons :

- It lacks precision

### MovingLeastSquare Transfer

This Transfer is based on m_src cell center positions and m_tgt cell centers positions. It is a "meshless" operation as it does not care about connectivity. There are several options :

- linear least square : the projection is the least square linear approx of the solution (can be an extrapolation)
- weighted least square : the projection is the weighted least square linear approx of the solution, there are several possibilities for the weighting function but it depends on the relative distance to the target interpolation point.